# LeetCode #1345: Jump Game IV

https://leetcode.com/problems/jump-game-iv/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS)** | $O(n^2)$ | $O(n)$ |
| **Optimal: BFS with Group Pruning ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (DFS)
Recursively try all three jump types with memoization. Without careful pruning, same-value groups can cause $O(n^2)$ work.

### Optimal: BFS with Group Pruning ★
Group all indices by value. BFS from index 0, expanding neighbors: $i-1$, $i+1$, and all same-value indices. When processing a same-value group, add all its members to the queue and then immediately clear the group — this prevents revisiting the group in future BFS levels, ensuring each index is enqueued at most once.

**Why this is better than Brute Force:** Group clearing guarantees $O(n)$ total work — each index is processed exactly once, and each group is visited at most once.

**Constraints:**
* $1 \le arr.length \le 5 \times 10^4$
* $-10^8 \le arr[i] \le 10^8$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
public class Solution {
    public int MinJumps(int[] arr) {
        int n = arr.Length;
        if (n == 1) return 0;
        // Group indices by value to enable O(1) same-value jumps
        var groups = new Dictionary<int, List<int>>();
        for (int i = 0; i < n; i++) {
            if (!groups.ContainsKey(arr[i])) groups[arr[i]] = new List<int>();
            groups[arr[i]].Add(i);
        }
        var visited = new bool[n];
        visited[0] = true;
        var queue = new Queue<int>();
        queue.Enqueue(0);
        int steps = 0;
        while (queue.Count > 0) {
            int size = queue.Count;
            steps++;
            while (size-- > 0) {
                int idx = queue.Dequeue();
                // Explore adjacent positions
                foreach (int next in new[] { idx - 1, idx + 1 }) {
                    if (next >= 0 && next < n && !visited[next]) {
                        if (next == n - 1) return steps;
                        visited[next] = true;
                        queue.Enqueue(next);
                    }
                }
                // Explore all same-value positions, then clear group to prevent revisits
                if (groups.TryGetValue(arr[idx], out var sameGroup)) {
                    foreach (int next in sameGroup) {
                        if (!visited[next]) {
                            if (next == n - 1) return steps;
                            visited[next] = true;
                            queue.Enqueue(next);
                        }
                    }
                    groups.Remove(arr[idx]); // clear so we never revisit this group
                }
            }
        }
        return -1;
    }
}

### Python

In [ ]:
from collections import deque, defaultdict
class Solution:
    def min_jumps(self, arr: list[int]) -> int:
        n = len(arr)
        if n == 1: return 0
        # Group indices by value to enable O(1) same-value jumps
        groups = defaultdict(list)
        for i, v in enumerate(arr): groups[v].append(i)
        visited = [False] * n
        visited[0] = True
        queue = deque([0])
        steps = 0
        while queue:
            steps += 1
            for _ in range(len(queue)):
                idx = queue.popleft()
                # Explore adjacent positions
                for nxt in (idx - 1, idx + 1):
                    if 0 <= nxt < n and not visited[nxt]:
                        if nxt == n - 1: return steps
                        visited[nxt] = True; queue.append(nxt)
                # Explore all same-value positions, then clear group to prevent revisits
                for nxt in groups.pop(arr[idx], []):
                    if not visited[nxt]:
                        if nxt == n - 1: return steps
                        visited[nxt] = True; queue.append(nxt)
        return -1

### Go

In [ ]:
func minJumps(arr []int) int {
    n := len(arr)
    if n == 1 { return 0 }
    // Group indices by value to enable O(1) same-value jumps
    groups := map[int][]int{}
    for i, v := range arr { groups[v] = append(groups[v], i) }
    visited := make([]bool, n)
    visited[0] = true
    queue := []int{0}
    steps := 0
    for len(queue) > 0 {
        steps++
        size := len(queue)
        for k := 0; k < size; k++ {
            idx := queue[k]
            // Explore adjacent positions
            for _, nxt := range []int{idx - 1, idx + 1} {
                if nxt >= 0 && nxt < n && !visited[nxt] {
                    if nxt == n-1 { return steps }
                    visited[nxt] = true; queue = append(queue, nxt)
                }
            }
            // Explore all same-value positions, then clear group to prevent revisits
            if sg, ok := groups[arr[idx]]; ok {
                for _, nxt := range sg {
                    if !visited[nxt] {
                        if nxt == n-1 { return steps }
                        visited[nxt] = true; queue = append(queue, nxt)
                    }
                }
                delete(groups, arr[idx])
            }
        }
        queue = queue[size:]
    }
    return -1
}

### Rust

In [ ]:
use std::collections::{VecDeque, HashMap};
impl Solution {
    pub fn min_jumps(arr: Vec<i32>) -> i32 {
        let n = arr.len();
        if n == 1 { return 0; }
        // Group indices by value to enable O(1) same-value jumps
        let mut groups: HashMap<i32, Vec<usize>> = HashMap::new();
        for (i, &v) in arr.iter().enumerate() { groups.entry(v).or_default().push(i); }
        let mut visited = vec![false; n];
        visited[0] = true;
        let mut queue = VecDeque::new();
        queue.push_back(0usize);
        let mut steps = 0;
        while !queue.is_empty() {
            steps += 1;
            let size = queue.len();
            for _ in 0..size {
                let idx = queue.pop_front().unwrap();
                // Explore adjacent positions
                for &nxt in &[idx.wrapping_sub(1), idx + 1] {
                    if nxt < n && !visited[nxt] {
                        if nxt == n-1 { return steps; }
                        visited[nxt] = true; queue.push_back(nxt);
                    }
                }
                // Explore all same-value positions, then clear group to prevent revisits
                if let Some(sg) = groups.remove(&arr[idx]) {
                    for nxt in sg {
                        if !visited[nxt] {
                            if nxt == n-1 { return steps; }
                            visited[nxt] = true; queue.push_back(nxt);
                        }
                    }
                }
            }
        }
        -1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [100,-23,-23,404,100,23,23,23,3,404]`
BFS from index 0 (value 100): level 1 visits indices 1 and 4 (same value). Level 2 from index 1 visits index 2; from index 4 visits 3 and 5. Eventually reaches index 9. Answer: **3**.

### 2. Slightly Complex
**Input:** `arr = [7,6,9,6,9,6,9,7]`
Value 7 appears at 0 and 7 (last). BFS level 1: same-value group for 7 immediately adds index 7 = last index. Answer: **1**.

### 3. Edge Case: Time Factor
**Input:** All $5 \times 10^4$ elements have the same value.
Level 1 of BFS visits index 1 (adjacent), then clears the entire same-value group — all $5 \times 10^4$ indices enqueued at once. Group clearing prevents re-visiting; total work is $O(n)$.

### 4. Edge Case: Space Factor
**Input:** $5 \times 10^4$ unique values.
Groups map has $n$ entries, visited array is $n$ booleans, queue holds at most $n$ indices. Space is $O(n)$.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [0, 0]`
Index 0 value=0: level 1 BFS visits index 1 (adjacent, also same-value). Index 1 = last. Answer: **1**.